# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Little-Master-Umer/FlyRank_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

## Lane: Refresh / Content Opportunity Scoring

Task type: Scoring

I frame Refresh / Content Opportunity as a **scoring** problem.

The goal is to assign each content item an opportunity score based on
multiple observed signals such as search demand, impressions, average
position, CTR, performance trend, content age, and time since the last
update.

A score is more useful than a simple yes/no decision because the content
team may have many items to review but limited time. The score can be used
to prioritize the items with the strongest signals first.

The output is therefore a decision-support score, not a claim that an item
definitely needs a refresh.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

print(f"Current working directory: {os.getcwd()}\n")
print("Contents of current directory:")
!ls -F

#Clone the GitHub repository
!git clone https://github.com/Little-Master-Umer/FlyRank_ML_Internship.git

#List the contents of the newly cloned directory to verify
print("\nContents of FlyRank_ML_Internship repository:")
!ls -F FlyRank_ML_Internship/

import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv("FlyRank_ML_Internship/data/raw/content_refresh_anonymized.csv")
# Check that the main signals needed for the scoring problem exist.

candidate_features = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_pct",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
]

available_features = [c for c in candidate_features if c in df.columns]
missing_features = [c for c in candidate_features if c not in df.columns]

print("Available candidate features:")
print(available_features)

print("\nMissing candidate features:")
print(missing_features)



Current working directory: /content

Contents of current directory:
FlyRank_ML_Internship/	sample_data/
fatal: destination path 'FlyRank_ML_Internship' already exists and is not an empty directory.

Contents of FlyRank_ML_Internship repository:
AGENTS.md  DATA_USE.md	LICENSE     README.md	      SETUP.md	   work/
CLAUDE.md  docs/	notebooks/  requirements.txt  skills/
data/	   GUIDE.md	outputs/    scripts/	      submission/
Available candidate features:
['search_volume', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'trend_pct', 'content_age_days', 'days_since_last_update', 'engagement_rate']

Missing candidate features:
[]


## 2. Target or proxy

The starter data does not contain an observed label such as
`refresh_needed` or `refresh_success`.

Therefore, I would not claim that the dataset contains a ground-truth
refresh target.

Instead, I would use a **defined proxy for refresh opportunity**.

The proxy would combine observable signals that can indicate potential
opportunity, including:

- existing search demand and impressions,
- weaker or changing search performance,
- declining trend percentage,
- older content,
- and time since the last update.

For example, content with meaningful search visibility but declining
performance may be more useful to review than content with no measurable
demand.

This proxy is a defined decision rule for experimentation, not an observed
business outcome. A future version of the system could replace it with an
observed outcome such as measured improvement after a content refresh.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

proxy_df = df.copy()

# Simple, transparent proxy components.
# These are deliberately directional rather than presented as ground truth.

proxy_df["has_search_demand"] = (
    proxy_df["search_volume"].fillna(0) > 0
).astype(int)

proxy_df["has_visibility"] = (
    proxy_df["impressions_90d"].fillna(0) > 0
).astype(int)

proxy_df["declining"] = (
    proxy_df["trend_pct"].fillna(0) < 0
).astype(int)

proxy_df["older_content"] = (
    proxy_df["content_age_days"].fillna(0) >= 365
).astype(int)

proxy_df["not_recently_updated"] = (
    proxy_df["days_since_last_update"].fillna(0) >= 90
).astype(int)

proxy_df["opportunity_proxy"] = (
    proxy_df["has_search_demand"]
    + proxy_df["has_visibility"]
    + proxy_df["declining"]
    + proxy_df["older_content"]
    + proxy_df["not_recently_updated"]
)

proxy_df[
    [
        "content_id",
        "search_volume",
        "impressions_90d",
        "trend_pct",
        "content_age_days",
        "days_since_last_update",
        "opportunity_proxy",
    ]
].head(10)

,content_id,search_volume,impressions_90d,trend_pct,content_age_days,days_since_last_update,opportunity_proxy
0,content_304f48230142,10.0,3803,-41.4,187,20,3
1,content_a1fb4e703a9e,90.0,15320,-57.7,445,25,4
2,content_9aa793d4d895,0.0,12581,-60.9,141,20,2
3,content_331d6c4de07b,10.0,11751,-13.8,463,22,4
4,content_d99b7a2d90ca,0.0,19140,-34.7,263,14,2
5,content_d4084a4bc775,720.0,3970,-38.9,147,20,3
6,content_9a34b442b552,0.0,20,-92.3,90,20,2
7,content_a63219c6e95a,590.0,1724,0.6,445,22,3
8,content_5e6c160719bc,0.0,32574,-58.8,90,20,2
9,content_c27558df2b0c,0.0,1240,-29.2,257,104,3


## 3. Success metric

### Metric: Precision@K

I would use **Precision@K** as the primary metric for the first version of
this scoring task.

The content team is unlikely to refresh every content item. Instead, they
may review a limited number of the highest-scored items.

Precision@K asks:

> Of the top K content items selected by the scoring system, how many
> satisfy the defined opportunity proxy?

For example, Precision@20 measures the proportion of the 20 highest-scored
items that meet the proxy definition.

A higher Precision@K means the top of the ranking contains more items that
match the defined refresh-opportunity criteria.

This is a decision-support metric. It does not prove that a refresh will
increase traffic or rankings because the starter data does not contain an
observed post-refresh outcome.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Example Precision@K calculation using the proxy.
# This is a framing check, not a trained-model evaluation.

def precision_at_k(scores, labels, k=20):
    order = np.argsort(-np.asarray(scores))[:k]
    selected_labels = np.asarray(labels)[order]
    return selected_labels.mean()

# Use the simple proxy as a temporary example score.
example_scores = proxy_df["opportunity_proxy"].fillna(0)
example_labels = (proxy_df["opportunity_proxy"] >= 3).astype(int)

k = min(20, len(proxy_df))

if k > 0:
    p_at_k = precision_at_k(example_scores, example_labels, k)
    print(f"Example Precision@{k}: {p_at_k:.3f}")
else:
    print("No rows available for Precision@K.")


Example Precision@20: 1.000


## 4. The unit of analysis, as a real dataframe

### Unit of analysis

**One row represents one content item for one client.**

The `content_id` identifies the content item and `client_id` identifies the
client associated with it.

The features in the row describe that content item's search performance,
engagement, age, freshness, and other observable characteristics.

This unit makes sense for the refresh use case because the eventual action
is also content-level: a content team can review and prioritize individual
content items.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the actual unit of analysis.

unit_columns = [
    "content_id",
    "client_id",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
]

unit_columns = [c for c in unit_columns if c in df.columns]

unit_df = df[unit_columns].copy()

print(f"Rows: {unit_df.shape[0]}")
print(f"Columns shown: {unit_df.shape[1]}")

unit_df.head(10)

Rows: 30000
Columns shown: 12


,content_id,client_id,search_volume,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct,content_age_days,days_since_last_update,engagement_rate
0,content_304f48230142,client_f369cb89fc,10.0,3803,29,0.76,10.6,down,-41.4,187,20,5.88
1,content_a1fb4e703a9e,client_4e07408562,90.0,15320,7,0.05,20.3,down,-57.7,445,25,0.00
2,content_9aa793d4d895,client_7f2253d7e2,0.0,12581,11,0.09,36.5,down,-60.9,141,20,0.00
3,content_331d6c4de07b,client_19581e27de,10.0,11751,58,0.49,6.2,stable,-13.8,463,22,1.28
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,19140,24,0.13,44.0,down,-34.7,263,14,0.00
5,content_d4084a4bc775,client_f369cb89fc,720.0,3970,1,0.03,8.5,down,-38.9,147,20,0.00
6,content_9a34b442b552,client_8722616204,0.0,20,0,0.00,7.0,down,-92.3,90,20,0.00
7,content_a63219c6e95a,client_19581e27de,590.0,1724,1,0.06,21.2,stable,0.6,445,22,3.57
8,content_5e6c160719bc,client_6208ef0f77,0.0,32574,29,0.09,46.0,down,-58.8,90,20,5.88
9,content_c27558df2b0c,client_19581e27de,0.0,1240,2,0.16,4.9,down,-29.2,257,104,0.00


## 5. Why ML beats a fixed rule here

A fixed rule could use one threshold, for example:

> If content is older than 365 days, refresh it.

That rule is easy to understand, but it ignores important differences between
content items.

Two older content items can have very different search visibility, average
position, CTR, engagement, and performance trends.

The dataset contains several signals that can interact. For example, a
content item can be old but still perform steadily, while another item can
have meaningful impressions and search demand but show a strong decline in
performance.

A scoring model can learn how multiple signals relate to the defined target
or proxy and produce a prioritized ranking rather than relying on one
manually chosen threshold.

However, ML is only justified if it performs better than a simple baseline.
I would therefore compare the ML scorer against transparent fixed-rule
baselines using the same evaluation metric.

The eventual action is decision-support: give the content team a ranked
shortlist of pages to review for a possible refresh.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show that the dataset contains variation across several signals.
# This supports the argument that a single threshold may lose information.

comparison_columns = [
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "trend_pct",
    "content_age_days",
    "days_since_last_update",
]

comparison_columns = [
    c for c in comparison_columns if c in df.columns
]

df[comparison_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
trend_pct,26612.0,-4.785969,473.861780,-100.0,-62.6,-33.50,0.00,44900.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.